# Session 10 — The model landscape: choosing a model
**Catalog · cost · speed · quality · the selection matrix**

**Introduction**

Different AI models have different strengths. Some are cheaper, some are faster, and others give better answers.

In this session, we will learn how to compare AI models and choose the best model for a particular task.

The main goal is to select a model based on its cost, speed, quality, and requirements.

## 1. Understanding Providers and Models

A provider is a service that gives us access to AI models, while a model is the AI system that performs a task.

We use Groq to access a hosted AI model and Ollama to run a model on our own computer.

Different models can have different costs, response times, and capabilities. Therefore, we should compare them before using them in a project.



In [1]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from landscape_utils import *
from tokens_utils import chat_client, ollama_client
ollama = ollama_client()
try:
    groq = chat_client()                      # Groq, from .env
except Exception as e:                        # no key yet: the notebook still runs; the hosted rows are just skipped
    groq = None
    print("no hosted client:", str(e)[:80])
GROQ_MODEL = os.getenv("MODEL")
print("hosted model:", GROQ_MODEL, "| expected EMI:", EXPECTED_EMI, "| expected ratio:", EXPECTED_RATIO, "%")

hosted model: openai/gpt-oss-20b | expected EMI: 17594.09 | expected ratio: 42.7 %


## 2. The catalog — ten rows, one table
A model catalog is a list of available models and their important details.

Some common terms in a model catalog are:-

- Open-weight: The model's learned weights are available to download.

- Self-hosted: The model runs on our own computer or server.

- Context window: The amount of information the model can process at one time.

- Input price: The cost of processing the input tokens.

- Output price: The cost of generating the output tokens.

- Quality: How well the model performs a task.

- Latency: How long the model takes to respond.

- Model prices and specifications can change, so we should verify them using the provider's official information.



In [2]:
cat = load_catalog()
print(f"{'model':46} {'access':11} {'open':5} {'ctx':>8} {'in$':>6} {'out$':>6} {'Q':>2} {'s':>4}")
for m in cat:
    pin = "?" if m["price_in"] is None else f"{m['price_in']:.3f}"
    pout = "?" if m["price_out"] is None else f"{m['price_out']:.2f}"
    print(f"{m['name'][:46]:46} {m['access']:11} {str(m['open_weight'])[:1]:5} {m['ctx_tokens']:8d} {pin:>6} {pout:>6} {m['quality']:2d} {m['latency_s']:4.1f}")

model                                          access      open       ctx    in$   out$  Q    s
gpt-oss-20b on Groq                            hosted-api  T       131072  0.075   0.30  3  1.5
gpt-oss-120b on Groq                           hosted-api  T       131072  0.150   0.60  4  2.0
Qwen3 32B on Groq                              hosted-api  T       131072  0.290   0.59  3  2.0
Llama 4 Scout 17B-16E on Groq                  hosted-api  T       131072  0.110   0.34  3  1.5
Llama 3.1 8B Instant on Groq                   hosted-api  T       131072  0.050      ?  2  1.0
llama3.2:3b on your laptop (Ollama)            local       T       131072  0.000   0.00  2  8.0
gpt-oss-20b self-hosted on a smaller GPU serve self-hosted T       131072  0.000   0.00  3  2.5
gpt-oss-120b self-hosted on your own GPU serve self-hosted T       131072  0.000   0.00  4  3.0
Frontier flagship (closed, vendor API)         vendor-api  F       200000  5.000  25.00  5  6.0
Frontier mid-tier (closed, vendor API)  

**Exercise 1 — fill the blank.** One row has `?` for output price. Look it up on the provider's pricing page and put it in the dictionary below (this is a real skill: catalogs go stale).
**Exercise 2 — verify two rows.** Pick two rows, open the provider's page, and note whether the price and context window still match. Write what you found in section 6.



In [3]:
OVERRIDES = {
    # "groq-llama31-8b": {"price_out": ???},      # <- exercise 1: replace ??? with the number you found
}

## 3. Three jobs, three bills
A loan assistant performs three different jobs. Each job has different requirements

- **faq** — Customers ask general questions about loans and banking products. The model should be affordable because it may handle many requests.
- **documents** — The model processes salary slips and bank statements. Since these documents contain private customer information, the data must stay inside the company's approved environment. **Customer PII: data must stay in-house.**
- **agent_assist** — The model provides suggestions while an employee speaks to a customer. It must respond quickly, ideally within about three seconds 

In [4]:
for key, sc in SCENARIOS.items():
    print(f"\n{key}: {sc['title']}  ({sc['calls_per_day']:,} calls/day, {sc['in_tokens']:,} in / {sc['out_tokens']} out per call)")
    for m in cat:
        m = {**m, **OVERRIDES.get(m["id"], {})}
        inr = monthly_cost_inr(m, sc)
        print(f"   {m['name'][:46]:46} " + ("price unknown" if inr is None else f"Rs {inr:>12,}/month"))


faq: Public product FAQ chatbot  (20,000 calls/day, 1,200 in / 200 out per call)
   gpt-oss-20b on Groq                            Rs        8,460/month
   gpt-oss-120b on Groq                           Rs       16,920/month
   Qwen3 32B on Groq                              Rs       26,282/month
   Llama 4 Scout 17B-16E on Groq                  Rs       11,280/month
   Llama 3.1 8B Instant on Groq                   price unknown
   llama3.2:3b on your laptop (Ollama)            Rs            0/month
   gpt-oss-20b self-hosted on a smaller GPU serve Rs       65,800/month
   gpt-oss-120b self-hosted on your own GPU serve Rs      188,000/month
   Frontier flagship (closed, vendor API)         Rs      620,400/month
   Frontier mid-tier (closed, vendor API)         Rs      248,160/month
   Frontier economy tier (closed, vendor API)     Rs       62,040/month

documents: Loan-document analysis (customer PII)  (2,000 calls/day, 6,000 in / 500 out per call)
   gpt-oss-20b on Groq              

## 4. Measure — the same prompt on a hosted model and on your laptop
Catalog information gives us an initial idea of a model's performance, but testing helps us make a better decision.

We can compare two models by sending them the same question and measuring their responses.

- **TTFT** — How long we wait before the first output token appears.
- **tokens/s** — How quickly the model generates output.
- **Total response time**: How long the complete response takes.
- **Answer quality**: Whether the answer is correct and follows the requirements.

In [5]:
print(PROMPT, "\n")
RESULTS = {}
for label, client, model in [("groq", groq, GROQ_MODEL), ("ollama", ollama, "llama3.2:3b")]:
    if client is None:
        print(f"--- {label}: skipped (no client - add your key to .env)\n"); continue
    try:
        r = time_stream(client, model)
        r["check"] = check_answer(r["text"])
        RESULTS[label] = r
        print(f"--- {label}: {model}")
        print(f"first token {r['ttft_s']} s | total {r['total_s']} s | {r['completion_tokens']} tokens | {r['tokens_per_s']} tokens/s")
        print("arithmetic check:", r["check"])
        print(r["text"].strip()[:500], "\n")
    except Exception as e:
        print(f"--- {label}: skipped ({type(e).__name__}: {str(e)[:120]})\n")

Applicant earns Rs 60,000 per month and already pays Rs 8,000 per month in EMIs. She asks for a Rs 8,00,000 personal loan at 11.5% per annum for 60 months. Compute the monthly EMI, then say whether (EMI + existing EMIs) is under 50% of her income. Answer in three short lines. 

--- groq: openai/gpt-oss-20b
first token 1.63 s | total 1.68 s | 720 tokens | 429.1 tokens/s
arithmetic check: {'emi_ok': True, 'ratio_ok': False, 'score_out_of_2': 1}
Monthly EMI ≈ Rs 17,600.  
Total payments (EMI + existing ₹8,000) ≈ Rs 25,600.  
This is below 50 % of her Rs 60,000 monthly income. 

--- ollama: llama3.2:3b
first token 8.1 s | total 22.19 s | 122 tokens | 5.5 tokens/s
arithmetic check: {'emi_ok': False, 'ratio_ok': False, 'score_out_of_2': 0}
To calculate the monthly EMI, we can use a loan calculator or the formula for monthly EMI:

Monthly EMI = Rs 8,00,000 / (11.5% * (1 - (1 + 0.115)^(-60))) ≈ Rs 9,119

Total EMIs = Rs 9,119 + Rs 8,000 = Rs 17,119.

The total EMIs exceed 50% of her income (Rs

**Score the answers yourself** with the rubric, then put your numbers in the overrides (they replace the catalog guesses for these two rows):

In [6]:
print(RUBRIC)

Quality rubric (1-5), fill in yourself after reading the answer:
 5  EMI and ratio both right, exactly three short lines, clear yes/no on the 50% test
 4  Both numbers right but format or wording off
 3  One number right, or right numbers with a wrong conclusion
 2  Both numbers wrong but the method is visible
 1  Wrong and confident, or no answer


In [7]:
# Fill in from YOUR run. total_s is a good stand-in for latency_s. quality: 1-5 from the rubric.
MEASURED = {
    # "groq-gpt-oss-20b":   {"latency_s": 1.4, "quality": 4},
    # "ollama-llama32-3b":  {"latency_s": 9.0, "quality": 2},
}
OVERRIDES.update(MEASURED)
OVERRIDES

{}

## 5. The selection matrix
A selection matrix helps us compare models systematically instead of choosing one based only on popularity.

We use three main steps:

**Step 1 — Check hard constraints**

First, remove models that do not meet essential requirements.

For example, a model that sends private customer data to an unapproved external service cannot be selected for a task that requires the data to remain in-house.

**Step 2 — Compare the remaining models**

Rank the models using factors such as cost, speed, and answer quality.

We can assign different weights to these factors depending on the task. A higher weight means that factor is more important.

**Step 3 — Test the selected model**

After selecting a model, test it on real examples from our project to confirm that it performs well.

A model that looks good on paper may not always be the best choice in practice.

In [8]:
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    print(f"\n=== {key}: {sc['title']}   weights {sc['weights']}")
    if not ranked:
        print("   nobody survives the hard constraints - relax one or add a model")
    for i, m in enumerate(ranked, 1):
        print(f"  {i}. {m['name'][:46]:46} score {m['score']:5.1f}   Rs {m['cost_inr']:>10,}/month   {m['latency_s']} s   Q{m['quality']}")
    for name, why in excluded:
        print(f"  x  {name[:46]:46} {'; '.join(why)}")


=== faq: Public product FAQ chatbot   weights {'cost': 0.6, 'latency': 0.2, 'quality': 0.2}
  1. gpt-oss-20b on Groq                            score  90.0   Rs      8,460/month   1.5 s   Q3
  2. Llama 4 Scout 17B-16E on Groq                  score  86.0   Rs     11,280/month   1.5 s   Q3
  3. gpt-oss-120b on Groq                           score  83.1   Rs     16,920/month   2.0 s   Q4
  4. Qwen3 32B on Groq                              score  72.0   Rs     26,282/month   2.0 s   Q3
  5. Frontier economy tier (closed, vendor API)     score  62.2   Rs     62,040/month   1.5 s   Q3
  6. gpt-oss-20b self-hosted on a smaller GPU serve score  57.0   Rs     65,800/month   2.5 s   Q3
  7. gpt-oss-120b self-hosted on your own GPU serve score  45.0   Rs    188,000/month   3.0 s   Q4
  8. Frontier mid-tier (closed, vendor API)         score  38.9   Rs    248,160/month   3.5 s   Q4
  9. Frontier flagship (closed, vendor API)         score  20.0   Rs    620,400/month   6.0 s   Q5
  x  Llama 3.1 8

## 6. Your decision — this is what you submit
For each task, we should select a suitable model, identify a second-best option, and explain our decision.

**faq:** Choose an affordable model that answers common questions correctly and quickly.

**documents:**  Choose a model that meets the company's data privacy requirements while providing acceptable accuracy.

**agent_assist:** Choose a model that responds quickly and gives reliable suggestions.

**Conclusion:-**

In this session, we learned that choosing an AI model requires more than comparing prices or selecting the most powerful model.

We should first check essential requirements, then compare cost, speed, and quality, and finally test the selected model on our own data.

The best model is the one that meets the task's requirements and provides the right balance of cost, speed, and quality.
Then save and submit: run the cell below, and upload `module01/s10_matrix_output.md` through the submission form (personal repositories start at Session 14).

In [9]:
from pathlib import Path
import landscape_utils
OUT = Path(landscape_utils.__file__).with_name("s10_matrix_output.md")   # next to the helper, whichever folder Jupyter runs in
lines = ["# Session 10 - selection matrix output\n", f"expected EMI {EXPECTED_EMI}, ratio {EXPECTED_RATIO}%\n"]
for key, sc in SCENARIOS.items():
    ranked, excluded = rank(sc, overrides=OVERRIDES)
    lines.append(f"\n## {key}: {sc['title']}")
    for i, m in enumerate(ranked, 1):
        lines.append(f"{i}. {m['name']} - score {m['score']} - Rs {m['cost_inr']:,}/month - {m['latency_s']} s - Q{m['quality']}")
    for name, why in excluded:
        lines.append(f"- excluded: {name} ({'; '.join(why)})")
lines.append("\n## Measured\n" + str({k: {kk: v[kk] for kk in ('ttft_s','total_s','tokens_per_s','check')} for k, v in RESULTS.items()}))
OUT.write_text("\n".join(lines), encoding="utf-8")
print("saved", OUT)

saved d:\genai-agents-course2\module01\s10_matrix_output.md
